In [ ]:
# !pip install -U --no-cache-dir accelerate peft bitsandbytes transformers trl

In [16]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    TrainingArguments, 
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import (
    PeftModel,
    LoraConfig, 
    get_peft_model, 
    TaskType, 
    prepare_model_for_kbit_training
)
import random
import warnings
# Filter out syntax warnings from libraries
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [3]:
# 1. Verify GPU is active
if not torch.cuda.is_available():
    raise ValueError("❌ GPU not detected! Check your Kaggle settings.")
print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")

✅ GPU Detected: Tesla P100-PCIE-16GB


In [4]:
# 1. Setup Model and Tokenizer
model_name = "google/flan-t5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# T5 is an Encoder-Decoder model, so we load it as such
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)
model.config.tie_word_embeddings = False

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
# 2. Prepare Data (Seq2Seq specific formatting)
# T5 needs inputs (Encoder) and targets (Decoder) separately.
data_files = {
    'train': '/kaggle/input/samsum-dataset-for-chat-summarization/train.json',
    'test': '/kaggle/input/samsum-dataset-for-chat-summarization/test.json'
}
dataset = load_dataset('json', data_files=data_files)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [6]:
def preprocess_function(examples):
    # Construct the prompt as the input
    inputs = [f"Summarize the following conversation:\n\n{dialogue}" for dialogue in examples['dialogue']]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    # Tokenize the targets (summaries)
    labels = tokenizer(text_target=examples['summary'], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [7]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/14732 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [8]:
# 3. Setup PEFT (LoRA)
# T5 requires TASK_TYPE="SEQ_2_SEQ_LM", not "CAUSAL_LM"
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, 
    inference_mode=False, 
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    bias="none"
)

In [9]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 344,064 || all params: 77,305,216 || trainable%: 0.4451


In [10]:
# 4. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="flan-t5-samsum-lora",
    learning_rate=2e-4,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,        # Print training loss every 10 steps (instead of 500)
    report_to="none",        # Print to the cell output (turns off WandB/Tensorboard)
    disable_tqdm=False,      # Force the progress bar to show
    
    # Optional: This enables generation during evaluation (good for summarization)
    predict_with_generate=True,
    dataloader_pin_memory=True
)

In [11]:
# 5. Trainer
# Use Seq2SeqTrainer instead of SFTTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,                  # Now passes the correct argument type
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,          # Updated from 'tokenizer'
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.852100,1.706551
2,1.821300,1.700825


TrainOutput(global_step=462, training_loss=1.874168213311728, metrics={'train_runtime': 478.0932, 'train_samples_per_second': 61.628, 'train_steps_per_second': 0.966, 'total_flos': 4404537877020672.0, 'train_loss': 1.874168213311728, 'epoch': 2.0})

In [13]:
# 1. Load the model you just trained
# (We use the model object currently in memory)
model.eval()

# 2. Pick a random sample from the test set

sample = dataset['test'][random.randint(0, len(dataset['test']))]
dialogue = sample['dialogue']
ground_truth = sample['summary']

# 3. Prepare input
input_text = f"Summarize the following conversation:\n\n{dialogue}"
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

# 4. Generate Summary
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"], 
        max_new_tokens=100, 
        do_sample=True, 
        top_p=0.9
    )

print("-" * 50)
print(f"INPUT DIALOGUE:\n{dialogue}")
print("-" * 50)
print(f"ACTUAL SUMMARY (Label):\n{ground_truth}")
print("-" * 50)
print(f"YOUR MODEL'S SUMMARY:\n{tokenizer.decode(outputs[0], skip_special_tokens=True)}")
print("-" * 50)

--------------------------------------------------
INPUT DIALOGUE:
Richie: Pogba
Clay: Pogboom
Richie: what a s strike yoh!
Clay: was off the seat the moment he chopped the ball back to his right foot
Richie: me too dude
Clay: hope his form lasts
Richie: This season he's more mature
Clay: Yeah, Jose has his trust in him
Richie: everyone does
Clay: yeah, he really deserved to score after his first 60 minutes
Richie: reward
Clay: yeah man
Richie: cool then 
Clay: cool
--------------------------------------------------
ACTUAL SUMMARY (Label):
Richie and Clay saw a very good football game, with one football player chopping the ball back to his foot, which was particularly exciting. Jose has trust in that player. 
--------------------------------------------------
YOUR MODEL'S SUMMARY:
Richie and Clay are excited to see Jose's form rewarded. He's more mature than Jose this season.
--------------------------------------------------


In [14]:
# 1. Save the adapter and tokenizer
output_path = "flan-t5-samsum-lora-final"
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

print(f"✅ Model saved to: {output_path}")

✅ Model saved to: flan-t5-samsum-lora-final


In [17]:
# 1. Setup paths
base_model_name = "google/flan-t5-small"
adapter_path = "flan-t5-samsum-lora-final" # This is the output_dir from the training step

# 2. Load the Base Model & Tokenizer
print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)

# 3. Load the LoRA Adapter
# This overlays your trained weights onto the base model
print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval() # Switch to evaluation mode

# 4. Define an Inference Function
def summarize_conversation(dialogue):
    # Format inputs just like we did during training
    input_text = f"Summarize the following conversation:\n\n{dialogue}"
    
    # Tokenize
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    # Generate Summary
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"], 
            max_new_tokens=100, 
            do_sample=True, 
            top_p=0.9
        )
    
    # Decode Output
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

# 5. Run it on a new sample
sample_dialogue = """
John: Hey, are we still on for the meeting at 3 PM?
Sarah: I might be a few minutes late. Traffic is terrible.
John: No worries. Should we start without you or wait?
Sarah: Please start without me. I'll join as soon as I can.
John: Okay, see you soon.
"""

print("-" * 30)
print(f"INPUT DIALOGUE:\n{sample_dialogue}")
print("-" * 30)
print(f"GENERATED SUMMARY:\n{summarize_conversation(sample_dialogue)}")
print("-" * 30)

Loading base model...
Loading LoRA adapter...
------------------------------
INPUT DIALOGUE:

John: Hey, are we still on for the meeting at 3 PM?
Sarah: I might be a few minutes late. Traffic is terrible.
John: No worries. Should we start without you or wait?
Sarah: Please start without me. I'll join as soon as I can.
John: Okay, see you soon.

------------------------------
GENERATED SUMMARY:
John, Sarah, John and Sarah are going to meet at 3 PM. They will meet as soon as he can.
------------------------------


In [ ]:
# 2. Zip the folder so it's easy to download
!zip -r flan_t5_lora.zip flan-t5-samsum-lora-final